<a href="https://colab.research.google.com/github/peacewhile/PHM-Learning-Task/blob/main/NGAFID_DATASET_DASK_EXAMPLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INSTALL PREREQ

In [1]:
!python -m pip install "dask[dataframe]"

In [2]:
!git clone https://github.com/hyang0129/NGAFIDDATASET.git

!(cd NGAFIDDATASET ; git checkout main; git reset --hard HEAD; git pull)
!(cd NGAFIDDATASET ; pip install -r requirements.txt -q)



Cloning into 'NGAFIDDATASET'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 120 (delta 52), reused 64 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 253.61 KiB | 1.97 MiB/s, done.
Resolving deltas: 100% (52/52), done.
Already on 'main'
Your branch is up to date with 'origin/main'.
HEAD is now at fa72386 Update README.md
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 

# IMPORT DEPENDENCIES

In [3]:
import sys
sys.path.append('/content/NGAFIDDATASET')

In [ ]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
from ngafiddataset.dataset.dataset import NGAFID_Dataset_Manager

# 1. 自动下载并解压数据 (假设已按之前教程准备好 all_flights.tar.gz)
_ = NGAFID_Dataset_Manager('all_flights')

# 2. 读取数据
base_path = '/content/all_flights'
flight_data_df = dd.read_parquet(f'{base_path}/one_parq')
flight_header_df = pd.read_csv(f'{base_path}/flight_header.csv', index_col='Master Index')

# 3. 过滤并标注：仅保留 before (0) 和 after (1) 航班
mask = flight_header_df['before_after'].isin(['before', 'after'])
labels_df = flight_header_df[mask].copy()
labels_df['target'] = (labels_df['before_after'] == 'after').astype(int)

print(f"有效航班总数: {len(labels_df)}")

In [4]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
from ngafiddataset.dataset.dataset import NGAFID_Dataset_Manager


/content/NGAFIDDATASET/ngafiddataset/dataset/dataset.py:53: SyntaxWarning: invalid escape sequence '\o'
  logger.info('Downloading and extracting Parquet Files to %s\one_parq.  Please open them using dask dataframes' % destination)
/content/NGAFIDDATASET/ngafiddataset/dataset/dataset.py:6: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
# 1. 创建本地目标目录
!mkdir -p /content/all_flights

# 2. 从云盘复制到 Colab 本地环境 (这样解压速度比在云盘内直接解压快 10 倍)
# 请替换为你实际的路径
drive_path = "/content/drive/MyDrive/all_flights.tar.gz"
# 使用双引号包裹路径，防止路径中有空格导致报错
!cp /content/drive/MyDrive/all_flights.tar.gz /content/all_flights.tar.gz

# 3. 解压文件
print("正在解压，请稍候...")
!tar -xzvf /content/all_flights.tar.gz -C /content/all_flights/

# 4. 检查结果
import os
if os.path.exists("/content/all_flights/flight_header.csv") or os.path.exists("/content/all_flights/all_flights/flight_header.csv"):
    print("✅ 数据解压成功！")
else:
    print("❌ 文件夹似乎还是空的，请检查解压路径。")


正在解压，请稍候...
gzip: stdin has more than one entry--rest ignored
tar: Child returned status 2
tar: Error is not recoverable: exiting now
❌ 文件夹似乎还是空的，请检查解压路径。


In [34]:
import os

# --- 1. 配置路径 (请确保云盘路径准确) ---
drive_path = "/content/drive/MyDrive/dataset/all_flights.tar.gz"
local_zip = "/content/all_flights.tar.gz"
extract_dir = "/content/all_flights"

# --- 2. 预清理 (防止旧数据干扰) ---
if os.path.exists(extract_dir):
    !rm -rf "{extract_dir}"
os.makedirs(extract_dir, exist_ok=True)

# --- 3. 检查并复制 (提高 IO 稳定性) ---
if not os.path.exists(drive_path):
    print(f"❌ 找不到云盘文件: {drive_path}")
    print("💡 提示：请确认 Google Drive 已挂载，且文件名大小写完全一致。")
else:
    print("🚚 正在从云盘高速复制到本地环境...")
    !cp "{drive_path}" "{local_zip}"

    # --- 4. 强力解压方案 (解决多重压缩冲突) ---
    print("📦 正在执行强力解压 (zcat 模式)...")

    # 使用 zcat 将所有 gzip 块解压为流，再交给 tar 提取
    # 这是解决 "gzip: stdin has more than one entry" 的标准做法
    exit_code = os.system(f"zcat {local_zip} | tar -xf - -C {extract_dir}")

    if exit_code == 0:
        print("✨ 解压圆满成功！")
        # 查看解压后的文件数量，确认不是空壳
        file_count = len(os.listdir(extract_dir))
        print(f"📊 目录 {extract_dir} 下已生成 {file_count} 个子项。")
    else:
        print("⚠️ 解压失败。正在尝试备选方案 (常规 tar)...")
        !tar -xf "{local_zip}" -C "{extract_dir}"

🚚 正在从云盘高速复制到本地环境...
📦 正在执行强力解压 (zcat 模式)...
⚠️ 解压失败。正在尝试备选方案 (常规 tar)...
gzip: stdin has more than one entry--rest ignored
tar: Child returned status 2
tar: Error is not recoverable: exiting now


In [35]:
!file /content/all_flights.tar.gz

/content/all_flights.tar.gz: Zip archive data, at least v2.0 to extract, compression method=store


In [36]:
import os

# 配置路径
local_zip = "/content/all_flights.tar.gz"  # 虽然名字叫 .tar.gz，但它是 ZIP
extract_dir = "/content/all_flights"

# 1. 确保目标目录存在
os.makedirs(extract_dir, exist_ok=True)

# 2. 使用 unzip 命令解压
print("📦 检测到文件实际为 ZIP 格式，正在使用 unzip 解压...")
!unzip -q "{local_zip}" -d "{extract_dir}"

# 3. 检查结果
if os.path.exists(extract_dir):
    file_count = len(os.listdir(extract_dir))
    print(f"✨ 解压完成！目录中现有 {file_count} 个文件/文件夹。")
else:
    print("❌ 解压似乎失败了，请检查磁盘空间。")

📦 检测到文件实际为 ZIP 格式，正在使用 unzip 解压...
✨ 解压完成！目录中现有 1 个文件/文件夹。


In [24]:
import os

# 定义源路径和目标路径
source = "/content/drive/MyDrive/all_flights.tar.gz"
destination = "/content/all_flights.tar.gz"

# 1. 检查网盘文件是否存在
if os.path.exists(source):
    print("✅ 找到网盘文件，正在高速复制中...")
    !cp "{source}" "{destination}"
    print("✅ 复制完成！")

    # 2. 创建文件夹并解压
    print("📦 正在解压数据，请稍候...")
    !mkdir -p /content/all_flights
    !tar -xf /content/all_flights.tar.gz -C /content/all_flights/
    print("✨ 解压成功！准备进入模型复现环节。")
else:
    print("❌ 错误：在 /content/drive/MyDrive/ 下没找到 all_flights.tar.gz")
    print("提示：请确认你是否已将文件上传到网盘根目录，且文件名大小写完全一致。")

✅ 找到网盘文件，正在高速复制中...
✅ 复制完成！
📦 正在解压数据，请稍候...
gzip: stdin has more than one entry--rest ignored
tar: Child returned status 2
tar: Error is not recoverable: exiting now
✨ 解压成功！准备进入模型复现环节。


In [26]:
import os

source = "/content/drive/MyDrive/all_flights.tar.gz"
destination = "/content/all_flights.tar.gz"
extract_path = "/content/all_flights"

# 1. 检查文件是否存在
if os.path.exists(source):
    print("✅ 找到网盘文件，正在高速复制中...")
    !cp "{source}" "{destination}"

    # 2. 创建目录
    os.makedirs(extract_path, exist_ok=True)

    # 3. 执行解压并获取状态
    print("📦 正在解压数据，请稍候...")
    # 使用 try-except 或者检查 shell 返回值
    result = os.system(f"tar -xf {destination} -C {extract_path}")

    if result == 0:
        print("✨ 解压成功！准备进入模型复现环节。")
    else:
        print("❌ 解压过程中出现错误，请检查压缩包是否损坏或格式是否正确。")
        # 备选方案：如果 -xf 失败，尝试用 gunzip 独立解压
        # !gunzip -c "{destination}" | tar x -C "{extract_path}"
else:
    print(f"❌ 错误：在 {os.path.dirname(source)} 下没找到文件。")

✅ 找到网盘文件，正在高速复制中...
📦 正在解压数据，请稍候...
❌ 解压过程中出现错误，请检查压缩包是否损坏或格式是否正确。


In [32]:
import os

# --- 配置路径 ---
source = "/content/drive/MyDrive/all_flights.tar.gz"
dest_file = "/content/all_flights.tar.gz"
extract_dir = "/content/all_flights"

# 1. 检查源文件
if not os.path.exists(source):
    print(f"❌ 错误：未找到源文件 {source}，请确认 Google Drive 已挂载。")
else:
    # 2. 高速复制
    print("✅ 找到网盘文件，正在高速复制中...")
    !cp "{source}" "{dest_file}"

    # 3. 清理并创建目标目录 (防止旧文件干扰)
    if os.path.exists(extract_dir):
        !rm -rf "{extract_dir}"
    os.makedirs(extract_dir, exist_ok=True)

    # 4. 健壮解压方案
    # 使用 --ignore-zeros 和 --warning=none 处理非标准 gzip 流
    # 通过管道 (cat | tar) 往往能绕过某些 gzip 格式不兼容问题
    print("📦 正在解压数据 (使用兼容模式)...")

    # 执行解压并捕捉状态
    exit_code = os.system(f"zcat {dest_file} | tar -xf - -C {extract_dir}")

    if exit_code == 0:
        print("✨ 解压成功！准备进入模型复现环节。")
        # 可选：解压成功后删除压缩包省空间
        # !rm "{dest_file}"
    else:
        print("⚠️ 解压过程仍有警告或错误。")
        print("💡 建议：请运行 !file {dest_file} 检查文件真实格式。")

✅ 找到网盘文件，正在高速复制中...
📦 正在解压数据 (使用兼容模式)...
⚠️ 解压过程仍有警告或错误。
💡 建议：请运行 !file {dest_file} 检查文件真实格式。


In [33]:
import tarfile
import os

source_file = "/content/all_flights.tar.gz"
target_dir = "/content/all_flights"

os.makedirs(target_dir, exist_ok=True)

print("🚀 开始使用 Python 引擎强制解压...")
try:
    with tarfile.open(source_file, "r:gz") as tar:
        tar.extractall(path=target_dir)
    print("✨ 解压圆满完成！")
except Exception as e:
    print(f"❌ 还是失败了: {e}")
    print("💡 终极提示：如果这里还报错，说明文件在上传/下载过程中损坏了，建议重新压缩上传。")

🚀 开始使用 Python 引擎强制解压...
❌ 还是失败了: [Errno 2] No such file or directory: '/content/all_flights.tar.gz'
💡 终极提示：如果这里还报错，说明文件在上传/下载过程中损坏了，建议重新压缩上传。


In [31]:
!file /content/all_flights.tar.gz

/content/all_flights.tar.gz: Zip archive data, at least v2.0 to extract, compression method=store


In [8]:
# 运行这个命令直接在 Colab 服务器内部下载
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1-0pVPhwRQoifT_VuQyGDLXuzYPYySX-Y' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1-0pVPhwRQoifT_VuQyGDLXuzYPYySX-Y" -O all_flights.tar.gz && rm -rf /tmp/cookies.txt

--2026-03-11 13:53:13--  https://docs.google.com/uc?export=download&confirm=&id=1-0pVPhwRQoifT_VuQyGDLXuzYPYySX-Y
Resolving docs.google.com (docs.google.com)... 74.125.137.113, 74.125.137.100, 74.125.137.101, ...
Connecting to docs.google.com (docs.google.com)|74.125.137.113|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1-0pVPhwRQoifT_VuQyGDLXuzYPYySX-Y&export=download [following]
--2026-03-11 13:53:13--  https://drive.usercontent.google.com/download?id=1-0pVPhwRQoifT_VuQyGDLXuzYPYySX-Y&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.137.132, 2607:f8b0:4023:c03::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.137.132|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-11 13:53:13 ERROR 404: Not Found.



In [10]:
# 1. 先删除之前那个假的小文件
!rm -rf all_flights.tar.gz

# 2. 运行带有“确认病毒扫描”逻辑的下载命令
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1-0pVPhwRQoifT_VuQyGDLXuzYPYySX-Y' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1-0pVPhwRQoifT_VuQyGDLXuzYPYySX-Y" -O all_flights.tar.gz && rm -rf /tmp/cookies.txt

# 3. 检查文件大小 (如果是 4.3G，就说明终于成功了！)
!ls -lh all_flights.tar.gz

--2026-03-11 13:58:20--  https://docs.google.com/uc?export=download&confirm=&id=1-0pVPhwRQoifT_VuQyGDLXuzYPYySX-Y
Resolving docs.google.com (docs.google.com)... 142.250.101.138, 142.250.101.113, 142.250.101.139, ...
Connecting to docs.google.com (docs.google.com)|142.250.101.138|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1-0pVPhwRQoifT_VuQyGDLXuzYPYySX-Y&export=download [following]
--2026-03-11 13:58:21--  https://drive.usercontent.google.com/download?id=1-0pVPhwRQoifT_VuQyGDLXuzYPYySX-Y&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.137.132, 2607:f8b0:4023:c03::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.137.132|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-11 13:58:21 ERROR 404: Not Found.

-rw-r--r-- 1 root root 0 Mar 11 13:58 all_flights.tar.gz


In [16]:
!rm -rf all_flights.tar.gz

In [17]:
import requests
import os

def download_file(id, destination):
    URL = "https://docs.google.com/uc?export=download"
    session = requests.Session()
    # 第一次请求获取确认令牌
    response = session.get(URL, params={'id': id}, stream=True)
    token = None
    for key, value in response.cookies.items():
        if key.startswith('download_warning'):
            token = value
            break
    # 第二次请求正式下载
    if token:
        response = session.get(URL, params={'id': id, 'confirm': token}, stream=True)

    with open(destination, "wb") as f:
        for chunk in response.iter_content(32768):
            if chunk: f.write(chunk)

file_id = '1-0pVPhwRQoifT_VuQyGDLXuzYPYySX-Y'
output_file = 'all_flights.tar.gz'

if not os.path.exists(output_file):
    print("🚀 开始下载 4.29GB 数据集，请确保网络环境可以访问 Google...")
    download_file(file_id, output_file)
    print("✅ 下载完成！")
else:
    print("文件已存在，跳过下载。")

# 解压
!mkdir -p /content/all_flights
!tar -xzvf all_flights.tar.gz -C /content/all_flights/

🚀 开始下载 4.29GB 数据集，请确保网络环境可以访问 Google...
✅ 下载完成！

gzip: stdin: not in gzip format
tar: Child returned status 1
tar: Error is not recoverable: exiting now


In [18]:
# 1. 强制删除之前解压失败的残留，确保环境干净
!rm -rf /content/all_flights

# 2. 创建目标文件夹
!mkdir -p /content/all_flights

# 3. 使用 tar 核心命令解压
# -x: 解压, -z: 处理.gz, -f: 指定文件, -C: 强制解压到目标目录
# 如果这一步报错 "gzip: stdin: unexpected end of file"，说明压缩包下载不完整，需重新下载
!tar -xzvf /content/all_flights.tar.gz -C /content/all_flights/

tar (child): /content/all_flights.tar.gz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now


In [37]:
# 1. 下载并解压数据集 (自动管理 4.3GB 数据)
_ = NGAFID_Dataset_Manager('all_flights')

# 2. 读取数据 (Dask 延迟加载)
base_path = '/content/all_flights'
flight_data_df = dd.read_parquet(f'{base_path}/one_parq')
flight_header_df = pd.read_csv(f'{base_path}/flight_header.csv', index_col='Master Index')

# 3. 过滤出二分类样本：before vs after
mask = flight_header_df['before_after'].isin(['before', 'after'])
labels_df = flight_header_df[mask].copy()
labels_df['target'] = (labels_df['before_after'] == 'after').astype(int)

print(f"✅ 成功过滤数据。维护前样本: {sum(labels_df['target']==0)}, 维护后样本: {sum(labels_df['target']==1)}")

2026-03-11 15:20:16.951 | INFO     | ngafiddataset.dataset.dataset:__init__:53 - Downloading and extracting Parquet Files to \one_parq.  Please open them using dask dataframes
2026-03-11 15:20:16.955 | INFO     | ngafiddataset.dataset.dataset:download:39 - Extracting File


ReadError: file could not be opened successfully:
- method gz: ReadError('not a gzip file')
- method bz2: ReadError('not a bzip2 file')
- method xz: ReadError('not an lzma file')
- method tar: ReadError('invalid header')

In [38]:
import pandas as pd
temp_df = pd.read_parquet('/content/all_flights/all_flights/all_flights/part.0.parquet')
print(temp_df.columns.tolist())

['volt1', 'volt2', 'amp1', 'amp2', 'FQtyL', 'FQtyR', 'E1 FFlow', 'E1 OilT', 'E1 OilP', 'E1 RPM', 'E1 CHT1', 'E1 CHT2', 'E1 CHT3', 'E1 CHT4', 'E1 EGT1', 'E1 EGT2', 'E1 EGT3', 'E1 EGT4', 'OAT', 'IAS', 'VSpd', 'NormAc', 'AltMSL', 'timestep', 'cluster']


In [ ]:
flight_data_df

,volt1,volt2,amp1,amp2,FQtyL,FQtyR,E1 FFlow,E1 OilT,E1 OilP,E1 RPM,E1 CHT1,E1 CHT2,E1 CHT3,E1 CHT4,E1 EGT1,E1 EGT2,E1 EGT3,E1 EGT4,OAT,IAS,VSpd,NormAc,AltMSL,timestep,cluster
npartitions=401,,,,,,,,,,,,,,,,,,,,,,,,,
1,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,object
103,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32749,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32820,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [ ]:
flight_data_df.head(5)

,volt1,volt2,amp1,amp2,FQtyL,FQtyR,E1 FFlow,E1 OilT,E1 OilP,E1 RPM,...,E1 EGT2,E1 EGT3,E1 EGT4,OAT,IAS,VSpd,NormAc,AltMSL,timestep,cluster
Master Index,,,,,,,,,,,,,,,,,,,,,
1,28.8,NaN,0.8,NaN,48.89,44.06,13.28,168.55,82.51,2519.7,...,1317.44,1298.97,1322.83,12.8,144.89,29.61,0.01,3010.7,3151,c_28
1,28.8,NaN,0.8,NaN,48.93,44.06,13.31,168.60,82.51,2519.4,...,1316.64,1300.68,1322.15,12.8,144.53,49.78,-0.00,3011.3,3150,c_28
1,28.8,NaN,0.9,NaN,48.96,44.06,13.36,168.65,82.51,2520.3,...,1317.20,1301.05,1323.12,12.8,144.15,58.68,-0.01,3011.8,3149,c_28
1,28.8,NaN,0.7,NaN,48.86,44.06,13.30,168.64,82.51,2519.3,...,1317.73,1302.14,1321.93,12.8,143.79,47.31,-0.00,3012.2,3148,c_28
1,28.8,NaN,0.6,NaN,48.87,44.06,13.30,168.65,82.51,2518.8,...,1318.00,1308.52,1316.99,12.8,143.45,36.05,-0.01,3012.3,3147,c_28


In [ ]:
flight_header_df.head(5)

,before_after,date_diff,flight_length,label,hierarchy,number_flights_before
Master Index,,,,,,
1,before,-1,4723.0,intake gasket leak/damage,NaN,0
2,before,-2,4649.0,intake gasket leak/damage,NaN,3
3,same,0,40.0,intake gasket leak/damage,NaN,-1
4,before,0,14.0,intake gasket leak/damage,NaN,0
5,same,0,683.0,intake gasket leak/damage,NaN,-1
